# NuExtract3 (GGUF) sur CPU : extraction JSON depuis un PDF ou un texte

Ce notebook :
1. installe les dépendances et **télécharge le modèle** `numind/NuExtract3-GGUF` (Q4_K_M) ;
2. charge le modèle **sur CPU uniquement** ;
3. extrait des données JSON selon **`schema_json`** (CV complet) ou **`schema_json_experience`** (expériences seules) ;
4. traite un **PDF** (avec **choix des pages**) ou un **texte brut** ;
5. mesure le **temps d'exécution** et la **consommation de RAM** à chaque étape.

> Exécutez les cellules dans l'ordre. Les cellules 1 à 5 ne se lancent qu'une fois par session.

## 1. Installation

In [ ]:
!pip install -q llama-cpp-python pymupdf jinja2 huggingface_hub psutil

## 2. Imports et outils de mesure (temps + RAM)

In [ ]:
import os, re, json, time, threading
import fitz, psutil
from huggingface_hub import hf_hub_download
from jinja2.sandbox import ImmutableSandboxedEnvironment
from llama_cpp import Llama, LlamaGrammar

PROC = psutil.Process(os.getpid())
N_THREADS = os.cpu_count()

def ram_mo():
    """RAM utilisée par ce processus Python (le modèle tourne dedans), en Mo."""
    return PROC.memory_info().rss / 1024**2

class SuiviRAM:
    """Mesure le pic de RAM pendant un bloc `with`, par échantillonnage toutes les 0,1 s."""
    def __init__(self, intervalle=0.1):
        self.intervalle = intervalle
    def __enter__(self):
        self.avant = ram_mo()
        self.pic = self.avant
        self._stop = threading.Event()
        self._th = threading.Thread(target=self._boucle, daemon=True)
        self._th.start()
        self.t0 = time.perf_counter()
        return self
    def _boucle(self):
        while not self._stop.is_set():
            self.pic = max(self.pic, ram_mo())
            time.sleep(self.intervalle)
    def __exit__(self, *exc):
        self.duree = time.perf_counter() - self.t0
        self._stop.set(); self._th.join()
        self.apres = ram_mo()
        self.pic = max(self.pic, self.apres)

RAM_DEPART = ram_mo()
print(f"CPU : {N_THREADS} coeurs | RAM machine : {psutil.virtual_memory().total/1024**3:.1f} Go")
print(f"RAM du notebook au départ : {RAM_DEPART:.0f} Mo")

## 3. Téléchargement du modèle

In [ ]:
REPO_ID  = "numind/NuExtract3-GGUF"
FICHIER  = "NuExtract3-Q4_K_M.gguf"   # Q4_K_M = bon compromis qualité/vitesse (éviter Q2_K)

t0 = time.perf_counter()
MODEL_PATH = hf_hub_download(repo_id=REPO_ID, filename=FICHIER)
print(f"Modèle : {MODEL_PATH}")
print(f"Taille : {os.path.getsize(MODEL_PATH)/1024**3:.2f} Go | Téléchargement : {time.perf_counter()-t0:.1f} s")

## 4. Chargement du modèle sur CPU

In [ ]:
N_CTX = 16384   # fenêtre de contexte (prompt + réponse). Plus grand = plus de RAM.

if "llm" in globals():
    del llm     # libère un ancien modèle si la cellule est relancée

with SuiviRAM() as m:
    llm = Llama(
        model_path=MODEL_PATH,
        n_ctx=N_CTX,
        n_gpu_layers=0,        # CPU uniquement
        n_threads=N_THREADS,
        n_batch=512,
        use_mmap=False,        # charge tout en RAM : mesure de RAM fiable
        verbose=False,
    )

RAM_MODELE = m.apres
print(f"⏱ Chargement : {m.duree:.1f} s")
print(f"💾 RAM après chargement : {m.apres:.0f} Mo (+{m.apres - m.avant:.0f} Mo pour le modèle + contexte)")

## 5. Moteur d'extraction NuExtract

- Le **gabarit (schéma)** est passé au chat template de NuExtract via le paramètre `template` (et non dans le message).
- Les balises `{% generation %}` propres à `transformers` sont retirées pour que Jinja standard puisse lire le template.
- Option **`USE_GRAMMAR`** : force une sortie JSON valide qui respecte exactement les clés du schéma.

In [ ]:
USE_GRAMMAR = True   # False = plus rapide sur certains CPU, mais JSON non garanti

def _raise(msg):
    raise Exception(msg)

_src = re.sub(r"\{%-?\s*(end)?generation\s*-?%\}", "", llm.metadata["tokenizer.chat_template"])
_env = ImmutableSandboxedEnvironment(trim_blocks=True, lstrip_blocks=True,
                                     extensions=["jinja2.ext.loopcontrols"])
_env.globals["raise_exception"] = _raise
CHAT_TPL = _env.from_string(_src)

# ---- Gabarit NuExtract -> JSON Schema (pour la grammaire) ----
_TYPES = {"verbatim-string", "string", "integer", "number", "date", "date-time",
          "time", "country", "currency", "email", "boolean"}

def _to_schema(t):
    if isinstance(t, dict):
        return {"type": "object",
                "properties": {k: _to_schema(v) for k, v in t.items()},
                "required": list(t.keys()), "additionalProperties": False}
    if isinstance(t, list):
        if len(t) == 1 and (isinstance(t[0], dict) or t[0] in _TYPES):
            return {"type": "array", "items": _to_schema(t[0])}      # tableau
        if len(t) == 1 and isinstance(t[0], list):
            return {"type": "array", "items": {"enum": t[0]}}      # multi-choix
        return {"enum": t + [None]}                                # choix unique
    base = {"integer": "integer", "number": "number", "boolean": "boolean"}.get(t, "string")
    return {"type": [base, "null"]}

def _parse_json(txt):
    txt = txt.strip().strip("`").removeprefix("json").strip()
    try:
        return json.loads(txt)
    except json.JSONDecodeError:
        debut, fin = txt.find("{"), txt.rfind("}")
        return json.loads(txt[debut:fin + 1])

def extraire(texte, schema, max_tokens=2048):
    """Extrait un JSON du texte selon le schéma. Renvoie (data, stats)."""
    t_total = time.perf_counter()
    prompt = CHAT_TPL.render(
        messages=[{"role": "user", "content": [{"type": "text", "text": texte}]}],
        template=json.dumps(schema, indent=4, ensure_ascii=False),
        enable_thinking=False,
        add_generation_prompt=True,
    )
    n_prompt = len(llm.tokenize(prompt.encode("utf-8")))
    if n_prompt + max_tokens > llm.n_ctx():
        raise ValueError(f"Trop long : {n_prompt} tokens de prompt + {max_tokens} de sortie "
                         f"> contexte {llm.n_ctx()}. Réduisez les pages ou augmentez N_CTX.")

    grammar = (LlamaGrammar.from_json_schema(json.dumps(_to_schema(schema)), verbose=False)
               if USE_GRAMMAR else None)

    with SuiviRAM() as m:
        out = llm.create_completion(prompt, max_tokens=max_tokens,
                                    temperature=0.2, grammar=grammar)

    data = _parse_json(out["choices"][0]["text"])
    n_out = out["usage"]["completion_tokens"]
    stats = {
        "caracteres_entree": len(texte),
        "tokens_prompt": n_prompt,
        "tokens_sortie": n_out,
        "temps_generation_s": round(m.duree, 2),
        "vitesse_tok_s": round(n_out / m.duree, 2) if m.duree else None,
        "temps_total_s": round(time.perf_counter() - t_total, 2),
        "ram_avant_mo": round(m.avant),
        "ram_pic_mo": round(m.pic),
        "ram_supplementaire_generation_mo": round(m.pic - m.avant),
        "fin": out["choices"][0]["finish_reason"],   # "length" = sortie coupée
    }
    return data, stats

def afficher(data, stats):
    print(json.dumps(data, indent=2, ensure_ascii=False))
    print("\n----- Mesures -----")
    print(f"⏱ Temps total      : {stats['temps_total_s']} s "
          f"(génération {stats['temps_generation_s']} s, {stats['vitesse_tok_s']} tok/s)")
    print(f"🔤 Tokens           : {stats['tokens_prompt']} en entrée | {stats['tokens_sortie']} en sortie")
    print(f"💾 RAM pic          : {stats['ram_pic_mo']} Mo "
          f"(+{stats['ram_supplementaire_generation_mo']} Mo pendant la génération)")
    print(f"💾 RAM totale NuExtract (modèle + pic) : {stats['ram_pic_mo'] - RAM_DEPART:.0f} Mo")
    if stats["fin"] == "length":
        print("⚠️ Sortie coupée : augmentez max_tokens.")

print("Moteur prêt ✅")

## 6. Schémas

Types NuExtract disponibles : `verbatim-string` (texte copié tel quel), `string` (texte reformulable), `date`, `email`, `integer`, `number`, `boolean`…
Un champ introuvable vaut `null`, une liste vide `[]`. Une liste de valeurs (`["A", "B"]`) impose un choix parmi ces valeurs.

In [ ]:
# ---- Schéma complet d'un CV ----
schema_json = {
    "nom_complet": "verbatim-string",
    "titre_profil": "string",
    "email": "email",
    "telephone": "verbatim-string",
    "adresse": "string",
    "linkedin": "verbatim-string",
    "resume": "string",
    "competences": ["string"],
    "langues": [
        {
            "langue": "string",
            "niveau": "string"
        }
    ],
    "formation": [
        {
            "diplome": "verbatim-string",
            "etablissement": "verbatim-string",
            "debut": "date",
            "fin": "date"
        }
    ],
    "experience": [
        {
            "poste": "verbatim-string",
            "entreprise": "verbatim-string",
            "debut": "date",
            "fin": "date",
            "description": "string"
        }
    ],
    "certifications": ["verbatim-string"]
}

# ---- Schéma expériences uniquement ----
schema_json_experience = {
    "experience": [
        {
            "poste": "verbatim-string",
            "entreprise": "verbatim-string",
            "debut": "date",
            "fin": "date",
            "description": "string"
        }
    ]
}

SCHEMAS = {"complet": schema_json, "experience": schema_json_experience}
print("Schémas :", list(SCHEMAS))

## 7. Outils PDF : choix des pages

In [ ]:
def parse_pages(spec, n_pages):
    """'1' | '1-2' | '1,3,5-6' | 'all' -> liste de numéros de page (à partir de 1)."""
    if str(spec).strip().lower() == "all":
        return list(range(1, n_pages + 1))
    pages = []
    for part in str(spec).split(","):
        part = part.strip()
        if "-" in part:
            a, b = map(int, part.split("-"))
            pages.extend(range(a, b + 1))
        elif part:
            pages.append(int(part))
    hors = [p for p in pages if not 1 <= p <= n_pages]
    if hors:
        raise ValueError(f"Pages {hors} hors limites (le PDF a {n_pages} pages)")
    return pages

def texte_pdf(pdf_path, pages="all"):
    with fitz.open(pdf_path) as doc:
        liste = parse_pages(pages, len(doc))
        texte = "\n".join(doc[p - 1].get_text() for p in liste)
        return texte, liste, len(doc)

def infos_pdf(pdf_path):
    """Affiche le nombre de tokens de chaque page pour choisir lesquelles extraire."""
    with fitz.open(pdf_path) as doc:
        total = 0
        for i, page in enumerate(doc, 1):
            n = len(llm.tokenize(page.get_text().encode("utf-8")))
            total += n
            vide = "  (vide : PDF scanné ?)" if not page.get_text().strip() else ""
            print(f"Page {i}: {n} tokens{vide}")
        print(f"Total : {total} tokens | contexte disponible : {llm.n_ctx()} "
              f"(garder ~2 500 pour le schéma et la réponse)")

print("Outils PDF prêts ✅")

## 8. Test sur un PDF

Réglez `PDF_PATH`, `PAGES` et `SCHEMA`.
Si le PDF est un scan (pages « vides »), il faut un OCR avant (dots.ocr, PaddleOCR-VL…) : NuExtract lit ici le texte natif du PDF.

In [ ]:
PDF_PATH = "/content/cv2.pdf"
PAGES    = "1-2"            # "1", "1-2", "1,3", "all"
SCHEMA   = "complet"        # "complet" ou "experience"

infos_pdf(PDF_PATH)

t0 = time.perf_counter()
cv_text, pages_lues, n_pages = texte_pdf(PDF_PATH, PAGES)
t_lecture = time.perf_counter() - t0
print(f"\nPages lues : {pages_lues} / {n_pages} | lecture PDF : {t_lecture:.2f} s\n")

resultat_pdf, stats_pdf = extraire(cv_text, SCHEMAS[SCHEMA])
stats_pdf.update({"fichier": PDF_PATH, "pages": pages_lues, "schema": SCHEMA,
                  "temps_lecture_pdf_s": round(t_lecture, 2)})
afficher(resultat_pdf, stats_pdf)

## 9. Test sur un texte brut

In [ ]:
TEXTE = """Mohamed Laanani
Data Scientist
mohamed.laanani@exemple.com | +212 6 12 34 56 78 | Casablanca

Expérience
Data Scientist — OCP Group, Casablanca (mars 2022 – aujourd'hui)
Conception de modèles de prévision de la demande et d'un pipeline MLOps.

Développeur stagiaire — Capgemini, Rabat (février 2021 – août 2021)
Développement d'une API REST en Python/FastAPI.

Formation
Master en Data Science — Université Hassan II (2019 – 2021)

Compétences : Python, SQL, PyTorch, Docker
Langues : Arabe (natif), Français (courant), Anglais (professionnel)
"""
SCHEMA = "complet"          # "complet" ou "experience"

resultat_texte, stats_texte = extraire(TEXTE, SCHEMAS[SCHEMA])
stats_texte.update({"source": "texte", "schema": SCHEMA})
afficher(resultat_texte, stats_texte)

## 10. Récapitulatif et sauvegarde

In [ ]:
lignes = []
for nom in ["stats_pdf", "stats_texte"]:
    if nom in globals():
        s = globals()[nom]
        lignes.append((nom.replace("stats_", ""), s["schema"], s["tokens_prompt"],
                       s["tokens_sortie"], s["temps_total_s"], s["ram_pic_mo"]))

print(f"{'Source':<8}{'Schéma':<12}{'Tok in':>8}{'Tok out':>9}{'Temps (s)':>11}{'RAM pic (Mo)':>14}")
for l in lignes:
    print(f"{l[0]:<8}{l[1]:<12}{l[2]:>8}{l[3]:>9}{l[4]:>11}{l[5]:>14}")
print(f"\nRAM après chargement du modèle : {RAM_MODELE:.0f} Mo "
      f"| RAM actuelle : {ram_mo():.0f} Mo | départ : {RAM_DEPART:.0f} Mo")

sortie = {}
if "resultat_pdf" in globals():
    sortie["pdf"] = {"resultat": resultat_pdf, "mesures": stats_pdf}
if "resultat_texte" in globals():
    sortie["texte"] = {"resultat": resultat_texte, "mesures": stats_texte}
with open("/content/resultats_nuextract.json", "w", encoding="utf-8") as f:
    json.dump(sortie, f, indent=2, ensure_ascii=False)
print("\nRésultats sauvegardés dans /content/resultats_nuextract.json")